# KB0 — Service time & arrival (walkthrough)

Chạy các lệnh từ thư mục gốc repo (hoặc chỉnh đường dẫn). Cần Prometheus port-forward và cluster đang phục vụ tải ổn định.

In [ ]:
import subprocess, sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
TCAL = os.path.join(ROOT, 'tools', 'calibration')
os.environ['PYTHONUTF8'] = '1'

## 1) E[S], Cs, μ từ Prometheus

Sửa `--prometheus-url` và `--matchers` cho đúng cluster.

In [ ]:
cmd = [
    sys.executable,
    os.path.join(TCAL, 'estimate_execution_stats.py'),
    '--prometheus-url', 'http://localhost:9090',
    '--matchers', 'namespace="nt531-env",app="processing"',
    '--window', '5m',
]
subprocess.run(cmd, cwd=TCAL, check=False)

## 2) Phân tích arrival (sau khi có `*_arrival_timestamps.csv` từ Locust)

Đặt `TIMESTAMPS_CSV` trỏ tới file thực tế.

In [ ]:
TIMESTAMPS_CSV = os.path.join(ROOT, 'src', 'load-generator', 'results', 'kb0_stable_arrival_arrival_timestamps.csv')
OUT_PNG = os.path.join(TCAL, 'out', 'arrival.png')
os.makedirs(os.path.dirname(OUT_PNG), exist_ok=True)
if os.path.isfile(TIMESTAMPS_CSV):
    subprocess.run([sys.executable, os.path.join(TCAL, 'analyze_arrivals.py'), '--timestamps', TIMESTAMPS_CSV, '--plot', OUT_PNG], cwd=TCAL, check=False)
else:
    print('Chưa có file timestamps — chạy Locust với EXPORT_ARRIVAL_TIMESTAMPS=1', TIMESTAMPS_CSV)